# Warning: Due to space constraints from running my tests I clear out the zip and unzipped folders. Ensure you have the zip stored in a separate location for additional operations.

# Note it was found that I could not easily dynamically pull the notebook name so if you update the notebook update the following code:

In [1]:
nb_name = "MAT-12-02-level-2-notebook"

In [2]:
# sudo apt install python3-dmidecode
#!pip install llama-cpp-python   --upgrade   --force-reinstall   --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu126
#!pip3 install torch==2.8.0+cu126 --index-url https://download.pytorch.org/whl/cu126

In [ ]:
export LD_LIBRARY_PATH=$CONDA_PREFIX/lib:$LD_LIBRARY_PATH

In [3]:
# Native
import concurrent.futures
from datetime import datetime
import hashlib
import os
from pathlib import Path
import shutil
import sys
import time
import zipfile

# Third Party
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
from tqdm import tqdm

/home/flaniganp/miniconda3/envs/my-python-buddy-notebook/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
# Get project base directory (one level up from current working directory)
base_dir = Path.cwd().parent

base_application_dir = base_dir / "base_application"

# Convert to absolute string path
base_application_dir = str(base_application_dir.resolve())

# Add to Python path if not already present
if base_application_dir not in sys.path:
    sys.path.append(base_application_dir)

print("base_application added to PATH:")
print(base_application_dir)

base_application added to PATH:
/home/flaniganp/Documents/my-python-buddy/base_application


In [5]:
from views_chat_utilities import SYSTEM_PROMPTS

In [6]:
# Generates a cleaned text response using a Hugging Face model pipeline (non-llama specific).
def get_cleaned_code_response_merged_model(llm_model, tokenizer, user_question):
    # Normalize system prompt
    system_prompt = SYSTEM_PROMPTS["code_writer"]

    # Build conversation structure
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_question},
    ]

    # Attempt to build a structured prompt (chat template if available)
    try:
        prompt = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )
    except Exception:
        bos = tokenizer.bos_token or ""
        prompt = f"{bos}[INST] <<SYS>>\n{system_prompt}\n<</SYS>>\n\n{user_question} [/INST]"

    # Configure model for inference
    llm_model.eval()
    try:
        # Some models may not support gradient checkpointing disable
        llm_model.gradient_checkpointing_disable()  # noqa: B110
    except AttributeError:
        # Attribute may not exist on all model types and is safe to ignore
        pass
    llm_model.config.use_cache = True

    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"

    # Generation settings
    max_new_tokens = 300
    generation_kwargs = dict(
        max_new_tokens=max_new_tokens,
        do_sample=False,
        repetition_penalty=1.05,
        no_repeat_ngram_size=6,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.eos_token_id,
        return_full_text=False,
    )

    # Create pipeline and generate
    text_pipe = pipeline(
        task="text-generation",
        model=llm_model,
        tokenizer=tokenizer,
        device_map="auto",
    )

    result = text_pipe(prompt, **generation_kwargs)

    # Extract generated text and token counts
    generated_text = result[0]["generated_text"].strip()
    prompt_tokens = len(tokenizer.encode(prompt))
    max_tokens_used = max_new_tokens

    # Postprocess text (placeholder for cleaning or formatting)
    cleaned_text = generated_text.strip()

    return cleaned_text, prompt_tokens, max_tokens_used

In [7]:
# Compute SHA256 of a file
def sha256sum(file_path, block_size=65536):
    sha = hashlib.sha256()
    with open(file_path, "rb") as f:
        for chunk in iter(lambda: f.read(block_size), b""):
            sha.update(chunk)
    return sha.hexdigest()

In [8]:
# Define paths
zip_path = os.path.join(base_dir, "models", "llama-2-7b-143k-codeAlpaca-2025-10-30_1326.zip")
hash_path = os.path.join(base_dir, "models", "llama-2-merged-7b-143k-codeAlpaca-2025-10-30_1326-hash.txt")
extract_dir = os.path.join(base_dir, "models", "llama-2-merged-7b-143k-codeAlpaca-2025-10-30_1143")

In [9]:
if os.path.exists(extract_dir):
    try:
        shutil.rmtree(extract_dir)
        print(f"Successfully deleted directory: {extract_dir}")
    except Exception as e:
        print(f"Error deleting {extract_dir}: {e}")
else:
    print(f"Directory not found: {extract_dir}.")

Directory not found: /home/flaniganp/Documents/my-python-buddy/models/llama-2-merged-7b-143k-codeAlpaca-2025-10-30_1143.


In [10]:
# Read expected hash
with open(hash_path, "r") as f:
    expected_hash = f.read().strip()

# Compute actual hash
actual_hash = sha256sum(zip_path)

# Compare
if actual_hash == expected_hash:
    print(f"Hash verified: {actual_hash}")
else:
    raise Exception((f"Hash mismatch!\nExpected: {expected_hash}\nFound: {actual_hash}"))

Hash verified: a7313874122da306d722fa8d77854a93cba398576f1a9b240830fbb6723e180b


In [11]:
if not os.path.exists(extract_dir):
    os.makedirs(extract_dir, exist_ok=True)
    with zipfile.ZipFile(zip_path, "r") as zf:
        members = zf.infolist()

        # Define a worker that extracts one file at a time
        def extract_member(member):
            zf.extract(member, extract_dir)

        # Use ThreadPoolExecutor (I/O bound) instead of ProcessPoolExecutor
        # because ZipFile objects aren’t pickle-safe
        with concurrent.futures.ThreadPoolExecutor() as executor:
            list(
                tqdm(
                    executor.map(extract_member, members),
                    total=len(members),
                    desc="Extracting",
                )
            )
    print(f"Extracted to: {extract_dir}")
else:
    print(f"Directory already exists: {extract_dir}")

Extracting: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████| 12/12 [00:21<00:00,  1.79s/it]

Extracted to: /home/flaniganp/Documents/my-python-buddy/models/llama-2-merged-7b-143k-codeAlpaca-2025-10-30_1143


In [12]:
# Remove archive (comment out if you want to keep it)
if os.path.exists(zip_path):
    try:
        os.remove(zip_path)
        print(f"Successfully deleted archive: {zip_path}")
    except Exception as e:
        print(f"Error deleting file {zip_path}: {e}")
else:
    print(f"File not found: {zip_path}")

Successfully deleted archive: /home/flaniganp/Documents/my-python-buddy/models/llama-2-7b-143k-codeAlpaca-2025-10-30_1326.zip


In [13]:
# Load model and tokenizer
print("Loading model...")
model = AutoModelForCausalLM.from_pretrained(extract_dir, device_map="auto")
tokenizer = AutoTokenizer.from_pretrained(extract_dir)

# Confirm load
print(f"Model and tokenizer loaded from {extract_dir}")

Loading model...


Loading checkpoint shards: 100%|██████████████████████████████████████████████████████████████████████████████████████████████| 3/3 [00:01<00:00,  1.51it/s]
Some parameters are on the meta device because they were offloaded to the cpu.


Model and tokenizer loaded from /home/flaniganp/Documents/my-python-buddy/models/llama-2-merged-7b-143k-codeAlpaca-2025-10-30_1143


In [14]:
questions = [
    "Code: How do I filter even numbers from a list using a list comprehension?",
    "Code: How do I find the common elements between two lists in Python?",
    "Code: How do I remove all vowels from a string in Python?",
    "Code: How do I count how many times each word appears in a string?",
    "Code: How do I check if a string is a palindrome, ignoring case and spaces?",
    "Code: How do I get the current date and time in Python?",
    "Code: How do I find the factorial of a number using recursion in Python?",
    "Code: How do I merge two dictionaries into one in Python?",
    "Code: How do I find all unique words in a sentence, ignoring case and punctuation?",
    "Code: How do I find the most frequent element in a list without using the collections module?",
    "Code: How do I determine if two strings are anagrams of each other in Python?",
]

In [15]:
# Setup output directory and file
base_dir = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()
results_dir = base_dir / "test_results"
results_dir.mkdir(exist_ok=True)

timestamp = datetime.now().strftime("%Y_%m_%d_%H_%M_%S")
output_file = results_dir / f"{nb_name}_{timestamp}.txt"

execution_times = []  # store all durations

# Begin loop over questions
with open(output_file, "w", encoding="utf-8") as f:
    for i, user_question in enumerate(questions, start=1):
        print(f"\nQuestion {i}: {user_question}")
        f.write(f"\nQuestion {i}: {user_question}\n")

        # --- Measure time ---
        start_time = time.time()
        cleaned_response, prompt_tokens, max_new_tokens = get_cleaned_code_response_merged_model(model, tokenizer, user_question)
        end_time = time.time()
        duration = end_time - start_time
        execution_times.append(duration)

        # --- Display and record results ---
        print(f"Cleaned Response:\n{cleaned_response}")
        print("-" * 88)

        f.write(f"Cleaned Response:\n{cleaned_response}\n")
        f.write(f"Prompt Tokens: {prompt_tokens}\n")
        f.write(f"Max New Tokens: {max_new_tokens}\n")
        f.write(f"Time Taken: {duration:.2f} seconds\n")
        f.write("Is result correct? If not, one sentence as to why:\n\n")
        f.write("-" * 88 + "\n")

    # Compute and record summary stats
    if execution_times:
        low_time = min(execution_times)
        high_time = max(execution_times)
        avg_time = sum(execution_times) / len(execution_times)

        summary = (
            f"\nExecution Time Summary:\n"
            f"Lowest Time:  {low_time:.2f} seconds\n"
            f"Highest Time: {high_time:.2f} seconds\n"
            f"Average Time: {avg_time:.2f} seconds\n"
        )

        print(summary)
        f.write(summary)

print(f"\nAll results recorded to: {output_file}")

Device set to use cuda:0
The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.



Question 1: Code: How do I filter even numbers from a list using a list comprehension?


Device set to use cuda:0


Cleaned Response:
```python
def filter_even(lst):
    """
    Filters even numbers from a list.
    
    Args:
        lst (list): List of numbers to filter.
        
    Returns:
        list: Filtered list of even numbers.
    """
    return [num for num in lst if num % 2 == 0]
```
----------------------------------------------------------------------------------------

Question 2: Code: How do I find the common elements between two lists in Python?


Device set to use cuda:0


...

Cleaned Response:
```python
def merge_dicts(d1, d2):
    """
    Merge two dictionaries into one.
    
    Args:
        d1 (dict): First dictionary to be merged.
        d2 (dict): Second dictionary to be merged. 
        
    Returns:
        dict: Merged dictionary.
    """
    merged_dict = {}
    for key, value in d1.items():
        merged_dict[key] = value
    for key, val in d2.items():
        if key not in merged_dict:
            merged_dict[key], val = val, key
    return merged_dict
```
----------------------------------------------------------------------------------------

Question 9: Code: How do I find all unique words in a sentence, ignoring case and punctuation?


Device set to use cuda:0


Cleaned Response:
```python
from typing import List

def get_unique_words(sentence: str) -> List[str]:
    """
    Returns a list of unique words in a sentence.
    
    Args:
        sentence (str): The sentence to analyze.
        
    Returns:
        List[str]: A list of unique words in the sentence.
    """
    # Initialize an empty list to store unique words
    unique_words = []
    
    # Split the sentence into individual words
    words = sentence.split()
    
    # Iterate over each word
    for word in words:
        # Convert the word to lowercase and remove punctuation
        cleaned_word = word.lower().replace('.', '').replace(',', '')
        
        # Check if the word is already in the list of unique words
        if cleaned_word not in unique_words:
            # Add the word to the list of unique words if it is not already present
            unique_words.append(cleaned_word)
    
    return unique_words
```
--------------------------------------------------------

Device set to use cuda:0


Cleaned Response:
```python
def most_frequent(lst):
    """Return the most frequent element in the list without using the collections.Counter module."""
    max_count = 0
    max_element = None
    for element in lst:
        count = lst.count(element)
        if count > max_count:
            max_count = count
            max_element = element
    return max_element
```
----------------------------------------------------------------------------------------

Question 11: Code: How do I determine if two strings are anagrams of each other in Python?
Cleaned Response:
```python
def are_anagrams(str1: str, str2: str) -> bool:
    """
    Returns True if the two strings are anagrames of each other, False otherwise.
    """
    # Remove non-alphanumeric characters from both strings
    str1 = ''.join(e for e in str1 if e.isalnum())
    str2 = ''.join(i for i in str2 if i.isalnum()) 

    # Sort the strings
    str1_sorted = sorted(str1)
    str2_sorted = sorted( str2 )

    # Check if the s

In [16]:
# Cleanup extracted directory
try:
    shutil.rmtree(extract_dir)
    print(f"Successfully deleted directory: {extract_dir}")
except Exception as e:
    print(f"Error deleting {extract_dir}: {e}")

Successfully deleted directory: /home/flaniganp/Documents/my-python-buddy/models/llama-2-merged-7b-143k-codeAlpaca-2025-10-30_1143
